In [ ]:
import numpy as np
from adaptive_latents import (
    StreamingKalmanFilter, StimRegressor, CenteringTransformer, proSVD, Pipeline, ArrayWithTime, datasets, Bubblewrap,
)
from adaptive_latents.regressions import BaseKernelRegressor
from adaptive_latents.stim_designer import StimDesigner
import matplotlib.pyplot as plt
from tqdm.auto import tqdm
import functools
import adaptive_latents.plotting_functions


In [ ]:
rng = np.random.default_rng()
d = datasets.Naumann24uDataset(1)

target_neurons = np.unique(d.opto_stimulations.target_neuron)

d.neural_data[np.isnan(d.neural_data)] = 0

def target_neuron_to_vector(tn):
    v = target_neurons * 0
    v[target_neurons == tn] = 1
    return v

d.opto_stimulations['stim_vector'] = d.opto_stimulations['target_neuron'].apply(target_neuron_to_vector)



In [ ]:
print(d.end_of_visual_period_time)
print(d.end_of_visual_period_sample)


In [ ]:
def make_sr(input_array, rng, stim_dict, autoreg=StreamingKalmanFilter, stim_rate=1 / 25, decay_rate=.9, prosvd_k=10, stim_magnitude=10, max_l0_norm=30, exit_time=np.inf, attempt_correction=True, heed_stimuli=True, stim_delay=0):
    sr = StimRegressor(
        autoreg=autoreg(),
        stim_designer=StimDesigner(max_l0_norm=max_l0_norm),
        stim_reg=BaseKernelRegressor(length_scale=0.011253, maxlen=20),
        # stim_reg=BaseKernelRegressor(length_scale=0.0017, maxlen=20),
        log_level=2,
        check_dt=True,
        attempt_correction=attempt_correction,
        heed_stimuli=heed_stimuli,
        stim_delay=stim_delay,
    )

    centerer = CenteringTransformer()
    pro = proSVD(k=prosvd_k)
    latents = []

    for data in Pipeline().streaming_run_on(input_array):

        if data.t in stim_dict:
            instantaneous_stim = stim_dict[data.t]
        else:
            instantaneous_stim = list(stim_dict.values())[0] * 0
        instantaneous_stim = ArrayWithTime(instantaneous_stim[None,:], data.t)


        data = centerer.partial_fit_transform(data, stream= 'X')
        data = pro.partial_fit_transform(data, stream='X')

        latents.append(data)

        sr.partial_fit_transform(instantaneous_stim, stream='stim')
        sr.partial_fit_transform(data, stream= 'X')

        if data.t > d.end_of_visual_period_time:
            centerer.freeze()
            pro.freeze()

        if data.t > exit_time:
            break
    sr.log['latents'] = ArrayWithTime.from_list(latents, squeeze_type='to_2d')

    return sr



In [ ]:
conditions = [
    dict(attempt_correction=True),
    dict(attempt_correction=False),
    dict(attempt_correction=False, heed_stimuli=False),
]



radius = 0
offsets = np.arange(-radius,radius+1) + 8
pred_errors = []
srs = []
s_sizes = offsets * np.nan


stims = ArrayWithTime(d.opto_stimulations['stim_vector'], d.opto_stimulations.time)
stim_dict = {k:v for k, v in zip(stims.t,stims)}
for idx, offset in enumerate(tqdm(offsets)):
    pred_errors.append([])
    srs.append([])
    for j, params in enumerate(conditions):
        sr = make_sr(input_array=d.neural_data, stim_dict=stim_dict, rng=rng,
                     # autoreg=functools.partial(
                     #     Bubblewrap,
                     #     num=200,
                     #     step=0.091,
                     #     eps=0.0581,
                     #     nu=0.0031,
                     #     M=745,
                     #     sigma_orig_adjustment=0,
                     #     # dead_nodes_unlikely=True,
                     #     log_level=2,
                     #     check_dt=True
                     # ),
                     autoreg=StreamingKalmanFilter,
                     stim_delay=offset*d.neural_data.dt, **params)
        pred_error = ArrayWithTime.from_list(sr.log['pred_error'], drop_early_nans=True, squeeze_type='to_2d')
        pred_errors[-1].append(pred_error)
        srs[-1].append(sr)



    # s_sizes[idx] = np.nanmean(sr.stim_reg.history[:,28:]**2)



In [ ]:
sr = make_sr(input_array=d.neural_data, stim_dict=stim_dict, rng=rng,
             autoreg=StreamingKalmanFilter,
             stim_delay=offset*d.neural_data.dt, **conditions[0])


In [ ]:
%matplotlib inline
adaptive_latents.plotting_functions.MultiRowRunComparison.compare_bw_runs([srs[0][0].autoreg, srs[0][2].autoreg])


In [ ]:
%matplotlib inline

s = slice(d.end_of_visual_period_time, None)

mses = [[ np.nanmean(pred_error.slice_by_time(s)**2) for pred_error in sub_l] for sub_l in pred_errors]
mses = np.array(mses).T

plt.plot(offsets, mses.T, '.-')
plt.ylabel('MSE')
plt.xlabel('delay')
plt.legend(['utilized', 'ignoring', 'unaware'])




In [ ]:
stim_reg: BaseKernelRegressor = srs[0][0].stim_reg

depth = 100
best_length_scale, (length_scales, errors, error_stds) = stim_reg.cross_validate_length_scale(np.logspace(-4,0, 40), depth=depth, ratio=.9)
error_sems = error_stds / np.sqrt(depth)


In [ ]:
fig, ax = plt.subplots()

ax.plot(length_scales, errors)
ax.fill_between(length_scales, errors - error_sems, errors + error_sems, alpha=0.2)
ax.axvline(best_length_scale, color='#dbb40c', linestyle='--')
ax.axvline(srs[0][0].stim_reg.length_scale, color='k')
ax.semilogx()
print(f'{best_length_scale=}')
print(f'current={sr.stim_reg.length_scale}')


In [ ]:

%matplotlib inline
fig, ax = plt.subplots(nrows=1, figsize=(18,4), layout='constrained')

pred_error = pred_errors[-1]


offset = 8



pred_error_comparison = [pred_errors[ np.nonzero(offsets == offset)[0][0] ][0], pred_errors[0][-1]]
pred_error_comparison.append(ArrayWithTime.from_list(sr.log['pred_error'], drop_early_nans=True, squeeze_type='to_2d'))


mse_comparison = [(pred_error**2).mean(axis=1) for pred_error in pred_error_comparison]


for mse, color, name in zip(mse_comparison, ['#ca1469ff', '#4d4d4dff'], ['utilized', 'unaware']):
    ax.plot(mse.t, mse, color=color, label=name)


mse_slice = mse.slice_by_time(slice(None, d.end_of_visual_period_time))
ax.plot(mse_slice.t, mse_slice, color='gray')


for t in d.opto_stimulations.time:
    ax.axvline(t, color='r')
ax.axvline(np.nan, color='r',  label='opto stimulations')

for t in d.visual_stimuli.time:
    ax.axvline(t, color='r', linestyle='--')
ax.axvline(np.nan, color='r', linestyle='--',  label='visual sitmuli')

for t in d.opto_stimulations.time:
    ax.axvline(t + (offset)*d.neural_data.dt, color='r', alpha=.25)
ax.axvline(np.nan, color='r',  label='time of correction', alpha=.25)




ax.set_ylim([-1, 10])
ax.set_xlim(left=d.end_of_visual_period_time - 100, right=1200)

ax.set_xlabel('time')
ax.set_ylabel('1-step MSE in latent space')

ax.legend()
fig.savefig('/home/jgould/Downloads/naumann_comparison.svg')


In [ ]:

print(f'utilized: {np.nanmean(mse_comparison[0].slice_by_time(s))}')
print(f'unaware:  {np.nanmean(mse_comparison[1].slice_by_time(s))}')
